<a href="https://colab.research.google.com/github/FrankChen0930/MarketMamba/blob/main/MarketMamba_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

print("🔌 正在掛載 Google Drive...")
drive.mount('/content/drive')

print("\n🚀 啟動 A100 極速武裝程序...")

# 1. 安裝圖神經網路核心套件 (PyTorch Geometric)
print("🕸️ 安裝 torch-geometric...")
!pip install -q torch-geometric

# 2. 尋找並安裝你編譯好的 Mamba 專武 (.whl)
# 假設你的 wheel 檔存在這個資料夾，如果路徑不同請幫我微調一下
WHEEL_DIR = '/content/drive/MyDrive/Mamba_Wheels_A100'

print(f"📦 從 {WHEEL_DIR} 載入預編譯套件...")
!pip install -q {WHEEL_DIR}/causal_conv1d-*.whl
!pip install -q {WHEEL_DIR}/mamba_ssm-*.whl

print("\n==========================================")
print("✅ Cell 1 執行完畢！")
print("A100 環境已完美配置 Mamba 與 GNN 套件！")
print("==========================================")

🔌 正在掛載 Google Drive...
Mounted at /content/drive

🚀 啟動 A100 極速武裝程序...
🕸️ 安裝 torch-geometric...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.9 MB/s eta 0:00:00
📦 從 /content/drive/MyDrive/Mamba_Wheels_A100 載入預編譯套件...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.8 MB/s eta 0:00:00

✅ Cell 1 執行完畢！
A100 環境已完美配置 Mamba 與 GNN 套件！


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

print("🗄️ 啟動 A100 宏觀時空資料載入器 (DataLoader)...\n")

TENSOR_PATH = '/content/drive/MyDrive/MarketMamba_Database/macro_aligned_tensor.pt'
EDGE_INDEX_PATH = '/content/drive/MyDrive/MarketMamba_Database/gnn_edge_index.pt'

class MacroSpatioTemporalDataset(Dataset):
    def __init__(self, tensor_path, seq_len=20):
        """
        seq_len: 模型一次回顧幾天的歷史 (預設 20 天 = 約一個月)
        """
        self.seq_len = seq_len

        print("📥 正在從 Google Drive 載入 3D 時空磚塊...")
        # 💡 解法就在這裡：加入 weights_only=False 允許讀取日期與字串
        data = torch.load(tensor_path, weights_only=False)
        self.X_full = data['X']  # 形狀: (8311, 1937, 5)
        self.y_full = data['y']  # 形狀: (8311, 1937)

        self.total_days = self.X_full.shape[0]
        self.num_stocks = self.X_full.shape[1]
        self.num_features = self.X_full.shape[2]

        # 可用的樣本數 = 總天數 - 回顧天數
        self.num_samples = self.total_days - self.seq_len
        print(f"✅ 載入成功！可切割出 {self.num_samples} 組訓練樣本。")

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # 截取長度為 seq_len 的歷史窗口
        # x_window 形狀: (Seq_Len, 1937, 5)
        x_window = self.X_full[idx : idx + self.seq_len]

        # 取窗口最後一天的未來報酬作為目標
        # y_target 形狀: (1937,)
        y_target = self.y_full[idx + self.seq_len - 1]

        return x_window, y_target

# ==========================================
# 🚀 載入測試與 GNN 網格準備
# ==========================================
# 1. 建立 Dataset 與 DataLoader
# batch_size=16 代表一次看 16 個不同的時間窗口
SEQ_LEN = 20
dataset = MacroSpatioTemporalDataset(TENSOR_PATH, seq_len=SEQ_LEN)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

# 2. 載入我們昨晚算好的 GNN 網格 (edge_index)
print("\n🕸️ 載入 GNN 全市場關聯矩陣...")
# 💡 這裡也一樣，因為裡面包了 tickers 字串，所以也要加 weights_only=False
gnn_data = torch.load(EDGE_INDEX_PATH, weights_only=False)
edge_index = gnn_data['edge_index']
print(f"🔗 成功載入 {edge_index.shape[1]} 條連線！")

# 3. 抽查第一個 Batch 的形狀
for batch_x, batch_y in train_loader:
    print("\n🔍 抽查第一個 Batch 的形狀：")
    # 注意：DataLoader 會在最前面自動加一個 Batch 維度
    print(f"📦 輸入特徵 X: {batch_x.shape} -> (Batch, Seq_Len, Stocks, Features)")
    print(f"🎯 預測目標 y: {batch_y.shape} -> (Batch, Stocks)")
    break

print("\n==========================================")
print("✅ Cell 2 執行完畢！資料管線已完美對接！")
print("==========================================")

🗄️ 啟動 A100 宏觀時空資料載入器 (DataLoader)...

📥 正在從 Google Drive 載入 3D 時空磚塊...
✅ 載入成功！可切割出 8291 組訓練樣本。

🕸️ 載入 GNN 全市場關聯矩陣...
🔗 成功載入 1390 條連線！

🔍 抽查第一個 Batch 的形狀：
📦 輸入特徵 X: torch.Size([32, 20, 1937, 5]) -> (Batch, Seq_Len, Stocks, Features)
🎯 預測目標 y: torch.Size([32, 1937]) -> (Batch, Stocks)

✅ Cell 2 執行完畢！資料管線已完美對接！


In [ ]:
import math
import torch
import torch.nn as nn
from mamba_ssm import Mamba
from torch_geometric.nn import GATConv

print("🧬 啟動 MarketGraphMambaDiffusion (防爆裝甲版)...\n")

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=time.device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class ConditionalDenoisingNet(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            TimeEmbedding(d_model),
            nn.Linear(d_model, d_model),
            nn.SiLU()
        )
        self.net = nn.Sequential(
            nn.Linear(1 + d_model + d_model, d_model * 2),
            nn.LayerNorm(d_model * 2), # 增加防爆層
            nn.SiLU(),
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),     # 增加防爆層
            nn.SiLU(),
            nn.Linear(d_model, 1)
        )

    def forward(self, noisy_target, time_step, context):
        batch_size, num_stocks = noisy_target.shape
        t_emb = self.time_mlp(time_step)
        t_emb = t_emb.unsqueeze(1).expand(-1, num_stocks, -1)
        noisy_target = noisy_target.unsqueeze(-1)
        x_input = torch.cat([noisy_target, context, t_emb], dim=-1)
        return self.net(x_input).squeeze(-1)

class MarketGraphMambaDiffusion(nn.Module):
    def __init__(self, input_dim=5, d_model=128, n_mamba_layers=4, gat_heads=8):
        super().__init__()
        self.d_model = d_model

        # 🛡️ 加入 LayerNorm 與 GELU 壓制初始雜訊
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU()
        )

        self.mamba_layers = nn.ModuleList([
            Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
            for _ in range(n_mamba_layers)
        ])
        # 🛡️ 專屬 Mamba 的正規化層
        self.mamba_norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(n_mamba_layers)
        ])

        self.gat = GATConv(in_channels=d_model, out_channels=d_model, heads=gat_heads, concat=False)
        # 🛡️ 專屬 GAT 的正規化層
        self.gat_norm = nn.LayerNorm(d_model)

        self.diffusion = ConditionalDenoisingNet(d_model=d_model)

    def forward(self, x_history, edge_index, noisy_target, time_step):
        batch_size, seq_len, num_stocks, _ = x_history.shape

        x = x_history.permute(0, 2, 1, 3).reshape(batch_size * num_stocks, seq_len, -1)
        x = self.embedding(x)

        # 🛡️ 實作殘差連接 (Residual Connection): x = x + Layer(Norm(x))
        for layer, norm in zip(self.mamba_layers, self.mamba_norms):
            x = x + layer(norm(x))

        node_features = x[:, -1, :].reshape(batch_size, num_stocks, self.d_model)

        gat_outs = []
        for i in range(batch_size):
            out = self.gat(node_features[i], edge_index)
            gat_outs.append(out)

        context = torch.stack(gat_outs, dim=0)
        # 🛡️ 壓制 GAT 交換情報後可能產生的數值膨脹
        context = self.gat_norm(context)
        context = torch.relu(context)

        predicted_noise = self.diffusion(noisy_target, time_step, context)
        return predicted_noise

# 重新部署至 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MarketGraphMambaDiffusion().to(device)
print(f"✅ 防爆版模型建構完畢！總訓練參數數量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} 個")

# 拿 DataLoader 的那一組真實資料來做前向傳播測試 (Forward Pass)
batch_x = batch_x.to(device)
edge_index = edge_index.to(device)
# 💡 動態抓取當前的 Batch Size
current_bsz = batch_x.shape[0]

# 模擬 Diffusion 訓練時的隨機雜訊與隨機時間步數 (改成 current_bsz)
dummy_noisy_target = torch.randn(current_bsz, 1937, device=device)
dummy_time_step = torch.randint(0, 100, (current_bsz,), device=device)

# 執行模型！
print("\n⚡ 正在進行張量維度壓力測試 (Forward Pass)...")
output_noise = model(batch_x, edge_index, dummy_noisy_target, dummy_time_step)

print(f"🎯 模型輸出形狀: {output_noise.shape} -> 完美符合預期 (Batch, Stocks)！")
print("\n==========================================")
print("✅ Cell 3 執行完畢！三位一體大腦運作正常！")
print("==========================================")

🧬 啟動 MarketGraphMambaDiffusion (防爆裝甲版)...

✅ 防爆版模型建構完畢！總訓練參數數量: 717,825 個

⚡ 正在進行張量維度壓力測試 (Forward Pass)...
🎯 模型輸出形狀: torch.Size([32, 1937]) -> 完美符合預期 (Batch, Stocks)！

✅ Cell 3 執行完畢！三位一體大腦運作正常！


In [ ]:
import gc
import torch

# 1. 殺掉持有計算圖的變數
if 'output_noise' in locals():
    del output_noise

# 2. 強制清空 PyTorch 的 GPU 暫存
torch.cuda.empty_cache()

# 3. 呼叫 Python 垃圾車
gc.collect()

print("🧹 空間已釋放！準備迎接正式訓練！")

🧹 空間已釋放！準備迎接正式訓練！


In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm.auto import tqdm

print("🔥 啟動 Mamba-GNN-Diffusion 聯合煉丹爐...\n")

# ==========================================
# 1. 擴散模型時程表 (Noise Scheduler)
# 負責控制「加雜訊」的力道
# ==========================================
class DDPMScheduler:
    def __init__(self, num_timesteps=100, device='cuda'):
        self.num_timesteps = num_timesteps
        self.device = device

        # 定義雜訊遞增的排程 (Beta Schedule)
        # 從 0.0001 慢慢增加到 0.02
        self.betas = torch.linspace(1e-4, 0.02, num_timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, original_samples, noise, timesteps):
        # 根據數學公式 q(x_t | x_0) 計算加噪後的結果
        sqrt_alpha_prod = torch.sqrt(self.alphas_cumprod[timesteps])
        sqrt_one_minus_alpha_prod = torch.sqrt(1 - self.alphas_cumprod[timesteps])

        # 增加維度以符合 (Batch, Stocks) 的廣播機制
        sqrt_alpha_prod = sqrt_alpha_prod.unsqueeze(-1)
        sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.unsqueeze(-1)

        # 原始資料 * 訊號保留比例 + 純雜訊 * 雜訊混入比例
        noisy_samples = sqrt_alpha_prod * original_samples + sqrt_one_minus_alpha_prod * noise
        return noisy_samples

# ==========================================
# 2. 訓練設定與優化器
# ==========================================
EPOCHS = 50  # 先跑 5 圈測試水溫
LEARNING_RATE = 1e-3
DIFFUSION_STEPS = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
edge_index = edge_index.to(device) # 確保 GNN 網格在 GPU 上

# 實例化排程器與優化器 (AdamW 是目前最強的優化器之一)
scheduler = DDPMScheduler(num_timesteps=DIFFUSION_STEPS, device=device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# ==========================================
# 3. 正式訓練迴圈 (Training Loop)
# ==========================================
print(f"🚀 開始訓練！總共 {EPOCHS} Epochs，每次預測 {1937} 檔股票的雜訊...")
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    # 用 tqdm 包裝 train_loader 來顯示進度條
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_x, batch_y in progress_bar:
        # 1. 將資料搬到 A100 上
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # 2. 清空前一步的梯度
        optimizer.zero_grad()

        # 3. 準備 Diffusion 的「真理 (Ground Truth)」
        # 產生與 batch_y 形狀完全一樣的純常態分配雜訊
        noise = torch.randn_like(batch_y)

        # 隨機為這個 Batch 裡的每一筆資料抽取一個時間步數 t (0 到 99)
        bsz = batch_y.shape[0]
        timesteps = torch.randint(0, scheduler.num_timesteps, (bsz,), device=device).long()

        # 將真實報酬率加上雜訊 (變成被污染的目標)
        noisy_y = scheduler.add_noise(batch_y, noise, timesteps)

        # 4. 前向傳播 (Forward Pass)
        # 讓 Mamba 看歷史、GNN 看連線，然後試圖猜出剛剛被加進去的是什麼雜訊
        predicted_noise = model(batch_x, edge_index, noisy_y, timesteps)

        # 5. 計算損失 (MSE: 預測的雜訊 vs 真實加進去的雜訊)
        loss = F.mse_loss(predicted_noise, noise)

        # 6. 反向傳播與權重更新
        loss.backward()
        # 梯度裁剪 (Gradient Clipping)，防止金融雜訊導致梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # 7. 更新進度條顯示的 Loss
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"📈 Epoch {epoch+1} 總結 | 平均 Loss: {avg_loss:.4f}")

    # 👇👇👇 加上這段保命符 👇👇👇
    SAVE_PATH = f'/content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth'
    torch.save(model.state_dict(), SAVE_PATH)
    print(f"💾 Epoch {epoch+1} 模型權重已安全備份至: {SAVE_PATH}")
    # 👆👆👆 加上這段保命符 👆👆👆

print("\n==========================================")
print("✅ Cell 4 訓練完畢！巨獸大腦已誕生！")
print("==========================================")

🔥 啟動 Mamba-GNN-Diffusion 聯合煉丹爐...

🚀 開始訓練！總共 50 Epochs，每次預測 1937 檔股票的雜訊...


Epoch 1/50:   0%|          | 0/259 [00:00<?, ?it/s]

📈 Epoch 1 總結 | 平均 Loss: 0.0937
💾 Epoch 1 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth


Epoch 2/50:   0%|          | 0/259 [00:00<?, ?it/s]

📈 Epoch 2 總結 | 平均 Loss: 0.0621
💾 Epoch 2 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth


...

📈 Epoch 25 總結 | 平均 Loss: 0.0407
💾 Epoch 25 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth


Epoch 26/50:   0%|          | 0/259 [00:00<?, ?it/s]

📈 Epoch 26 總結 | 平均 Loss: 0.0377
💾 Epoch 26 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth


Epoch 27/50:   0%|          | 0/259 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm.auto import tqdm

print("🔥 啟動手動排檔：MarketMamba 終極微調衝刺...\n")

# ==========================================
# 1. 喚醒沉睡的巨獸 (讀取剛才的存檔)
# ==========================================
SAVE_PATH = '/content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth'

# 將權重載入模型
model.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
print(f"✅ 成功載入 Epoch 26 的大腦記憶！")

# ==========================================
# 2. 學習率降檔 (Learning Rate Decay)
# ==========================================
# 將學習率從 1e-3 降到 1e-4，用小碎步精雕細琢
NEW_LR = 1e-4
optimizer = AdamW(model.parameters(), lr=NEW_LR, weight_decay=1e-4)
print(f"⚙️ 優化器已換檔！目前學習率: {NEW_LR}")

# ==========================================
# 3. 接續馬拉松 (Epoch 27 ~ 50)
# ==========================================
START_EPOCH = 26  # 已經跑完 26 圈，從第 27 圈 (索引 26) 開始
TOTAL_EPOCHS = 50

model.train()
print(f"\n🚀 重新點火！開始執行 Epoch {START_EPOCH+1} 到 {TOTAL_EPOCHS}...")

for epoch in range(START_EPOCH, TOTAL_EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}")

    for batch_x, batch_y in progress_bar:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        noise = torch.randn_like(batch_y)
        bsz = batch_y.shape[0]
        timesteps = torch.randint(0, scheduler.num_timesteps, (bsz,), device=device).long()
        noisy_y = scheduler.add_noise(batch_y, noise, timesteps)

        predicted_noise = model(batch_x, edge_index, noisy_y, timesteps)
        loss = F.mse_loss(predicted_noise, noise)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"📈 Epoch {epoch+1} 總結 | 平均 Loss: {avg_loss:.4f}")

    # 繼續保持安全存檔的優良傳統！
    torch.save(model.state_dict(), SAVE_PATH)
    print(f"💾 Epoch {epoch+1} 模型權重已安全備份至: {SAVE_PATH}")

print("\n==========================================")
print("🏆 終極馬拉松完賽！巨獸大腦完全體誕生！")
print("==========================================")

🔥 啟動手動排檔：MarketMamba 終極微調衝刺...

✅ 成功載入 Epoch 26 的大腦記憶！
⚙️ 優化器已換檔！目前學習率: 0.0001

🚀 重新點火！開始執行 Epoch 27 到 50...


Epoch 27/50:   0%|          | 0/259 [00:00<?, ?it/s]

📈 Epoch 27 總結 | 平均 Loss: 0.0349
💾 Epoch 27 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth


Epoch 28/50:   0%|          | 0/259 [00:00<?, ?it/s]

📈 Epoch 28 總結 | 平均 Loss: 0.0361
💾 Epoch 28 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth


...

📈 Epoch 48 總結 | 平均 Loss: 0.0350
💾 Epoch 48 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth


Epoch 49/50:   0%|          | 0/259 [00:00<?, ?it/s]

📈 Epoch 49 總結 | 平均 Loss: 0.0351
💾 Epoch 49 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth


Epoch 50/50:   0%|          | 0/259 [00:00<?, ?it/s]

📈 Epoch 50 總結 | 平均 Loss: 0.0360
💾 Epoch 50 模型權重已安全備份至: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth

🏆 終極馬拉松完賽！巨獸大腦完全體誕生！


In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error

print("📊 啟動 MarketMamba 回測與準確度評估中心...\n")

# 核心降噪函數保持不變
@torch.no_grad()
def generate_future_distributions(model, scheduler, x_latest, edge_index, num_futures=30):
    model.eval()
    device = x_latest.device
    num_stocks = x_latest.shape[2]

    x_history_expanded = x_latest.expand(num_futures, -1, -1, -1)
    x_t = torch.randn(num_futures, num_stocks, device=device)

    for i in tqdm(reversed(range(scheduler.num_timesteps)), total=scheduler.num_timesteps, desc="🌌 回測降噪生成中"):
        t = torch.full((num_futures,), i, device=device, dtype=torch.long)
        predicted_noise = model(x_history_expanded, edge_index, x_t, t)

        alpha_t = scheduler.alphas[i]
        alpha_t_cumprod = scheduler.alphas_cumprod[i]
        beta_t = scheduler.betas[i]

        if i > 0:
            noise = torch.randn_like(x_t)
        else:
            noise = torch.zeros_like(x_t)

        x_t = (1 / torch.sqrt(alpha_t)) * (x_t - ((1 - alpha_t) / torch.sqrt(1 - alpha_t_cumprod)) * predicted_noise) + torch.sqrt(beta_t) * noise

    return x_t

# ==========================================
# 🎯 歷史對答案：挑選已知未來的某一天
# ==========================================
# 我們拿資料集倒數第 10 筆來考它 (這代表我們握有真實的未來 5 天答案)
TEST_INDEX = len(dataset) - 100
test_x, true_y = dataset[TEST_INDEX]

test_x = test_x.unsqueeze(0).to(device) # (1, 20, 1937, 5)
true_y = true_y.numpy()                 # (1937,) 這就是真實發生的 5 日報酬率 (解答)

print(f"⏳ 正在對歷史時間點進行預測，準備對答案...")
future_returns = generate_future_distributions(model, scheduler, test_x, edge_index, num_futures=30)

# 用 30 個平行宇宙的平均值當作最終預測
pred_y = future_returns.mean(dim=0).cpu().numpy()

# ==========================================
# 📈 產出模型成績單
# ==========================================
# 1. 誤差計算
mae = mean_absolute_error(true_y, pred_y)

# 2. 漲跌方向勝率
true_direction = true_y > 0
pred_direction = pred_y > 0
accuracy = np.mean(true_direction == pred_direction)

# 3. 實戰選股模擬
tensor_data = torch.load(TENSOR_PATH, weights_only=False)
tickers = tensor_data['tickers']

df_eval = pd.DataFrame({
    'Ticker': tickers,
    'Predicted_Return': pred_y,     # 模型猜的
    'True_Return': true_y           # 真實發生的
})

# 👇 補上這一行：計算斯皮爾曼等級相關係數 (Rank IC)
ic_value = df_eval['Predicted_Return'].corr(df_eval['True_Return'], method='spearman')

# 照模型預測挑出最強的 10 檔
top_10_picks = df_eval.sort_values('Predicted_Return', ascending=False).head(10)
# 看看這 10 檔「實際上」賺了多少
actual_profit = top_10_picks['True_Return'].mean()

print("\n==========================================")
print("📝 【MarketMamba 訓練成果總結報告】 📝")
print("==========================================")
print(f"🥇 資訊係數 (Rank IC): {ic_value:.4f}  <-- 終極指標！")
print(f"📏 全市場平均絕對誤差 (MAE): {mae:.4f}")
print(f"🎯 漲跌方向預測準確率:      {accuracy * 100:.2f}%")
print("------------------------------------------")
print("🏆 【如果當時跟單 Top 10 選股，真實損益結果】")
print(top_10_picks.to_string(index=False))
print("------------------------------------------")
print(f"💰 實戰模擬報酬率: {actual_profit * 100:.2f}% (5天期間)")
print("==========================================")



📊 啟動 MarketMamba 回測與準確度評估中心...

⏳ 正在對歷史時間點進行預測，準備對答案...


🌌 回測降噪生成中:   0%|          | 0/100 [00:00<?, ?it/s]


📝 【MarketMamba 訓練成果總結報告】 📝
🥇 資訊係數 (Rank IC): 0.0268  <-- 終極指標！
📏 全市場平均絕對誤差 (MAE): 0.0361
🎯 漲跌方向預測準確率:      48.48%
------------------------------------------
🏆 【如果當時跟單 Top 10 選股，真實損益結果】
Ticker  Predicted_Return  True_Return
  2491          0.079773    -0.049159
  3532          0.062163    -0.167054
  1717          0.050615    -0.024754
  6781          0.048477    -0.129491
  3665          0.047680    -0.124984
  6419          0.046610    -0.077324
  5469          0.045280    -0.096850
  3704          0.042547    -0.065478
  3016          0.041744     0.033307
  6870          0.041377    -0.193291
------------------------------------------
💰 實戰模擬報酬率: -8.95% (5天期間)


In [ ]:
class DDPMScheduler:
    def __init__(self, num_timesteps=100, device='cuda'):
        self.num_timesteps = num_timesteps
        self.device = device

        # 定義雜訊遞增的排程 (Beta Schedule)
        # 從 0.0001 慢慢增加到 0.02
        self.betas = torch.linspace(1e-4, 0.02, num_timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, original_samples, noise, timesteps):
        # 根據數學公式 q(x_t | x_0) 計算加噪後的結果
        sqrt_alpha_prod = torch.sqrt(self.alphas_cumprod[timesteps])
        sqrt_one_minus_alpha_prod = torch.sqrt(1 - self.alphas_cumprod[timesteps])

        # 增加維度以符合 (Batch, Stocks) 的廣播機制
        sqrt_alpha_prod = sqrt_alpha_prod.unsqueeze(-1)
        sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.unsqueeze(-1)

        # 原始資料 * 訊號保留比例 + 純雜訊 * 雜訊混入比例
        noisy_samples = sqrt_alpha_prod * original_samples + sqrt_one_minus_alpha_prod * noise
        return noisy_samples

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error

print("🔌 啟動 MarketMamba 冷啟動評估中心...\n")

# ==========================================
# 0. 喚醒雲端硬碟裡的巨獸大腦
# ==========================================
SAVE_PATH = '/content/drive/MyDrive/MarketMamba_Database/MarketMamba_Giant_Latest.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 確保你的模型實例 (model) 在前面的 Cell 已經被定義了
try:
    model.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
    model.to(device)
    model.eval() # 鎖定權重，切換到推論模式
    print("✅ 成功從雲端硬碟載入 MarketMamba 訓練權重！")
except Exception as e:
    print(f"❌ 載入失敗，請確認前面的 Cell 是否已經執行了模型定義！錯誤訊息: {e}")

# ==========================================
# 0.5 補上失憶的零件：定義降噪排程器 (Scheduler)
# ==========================================
class DDPMScheduler:
    def __init__(self, num_timesteps=100, beta_start=0.0001, beta_end=0.02, device="cpu"):
        self.num_timesteps = num_timesteps
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

# 實例化 Scheduler，並丟到 GPU (或 CPU) 上
scheduler = DDPMScheduler(num_timesteps=100, device=device)

# 確保 edge_index 也有被定義 (通常是從你的 dataset 或是前面處理好的變數拿)
# 如果你本來的程式是用 dataset.edge_index，請確保加上這行：
if 'edge_index' not in locals():
    # 假設你的 edge_index 存在 dataset 裡，或者在前面的 Cell 已經跑過
    # 這裡做個防呆提示
    print("⚠️ 提醒：請確保你的 edge_index (GNN 關係網) 已經在記憶體中！")
    # edge_index = dataset.edge_index.to(device)  <-- 根據你原本的寫法可能需要這行

# ==========================================
# 1. 核心降噪函數
# ==========================================
@torch.no_grad()
def generate_future_distributions(model, scheduler, x_latest, edge_index, num_futures=30):
    model.eval()
    device = x_latest.device
    num_stocks = x_latest.shape[2]

    x_history_expanded = x_latest.expand(num_futures, -1, -1, -1)
    x_t = torch.randn(num_futures, num_stocks, device=device)

    for i in tqdm(reversed(range(scheduler.num_timesteps)), total=scheduler.num_timesteps, desc="🌌 回測降噪生成中"):
        t = torch.full((num_futures,), i, device=device, dtype=torch.long)
        predicted_noise = model(x_history_expanded, edge_index, x_t, t)

        alpha_t = scheduler.alphas[i]
        alpha_t_cumprod = scheduler.alphas_cumprod[i]
        beta_t = scheduler.betas[i]

        if i > 0:
            noise = torch.randn_like(x_t)
        else:
            noise = torch.zeros_like(x_t)

        x_t = (1 / torch.sqrt(alpha_t)) * (x_t - ((1 - alpha_t) / torch.sqrt(1 - alpha_t_cumprod)) * predicted_noise) + torch.sqrt(beta_t) * noise

    return x_t

# ==========================================
# 🎯 2. 歷史對答案：挑選 100 天前的時空節點
# ==========================================
# 聽你的，我們往前推 100 天來測試！
TEST_INDEX = len(dataset) - 100
test_x, true_y = dataset[TEST_INDEX]

test_x = test_x.unsqueeze(0).to(device)
true_y = true_y.numpy()

print(f"\n⏳ 正在預測 {TEST_INDEX} 號節點 (約 100 個交易日前) 的平行未來...")
future_returns = generate_future_distributions(model, scheduler, test_x, edge_index, num_futures=30)

pred_y = future_returns.mean(dim=0).cpu().numpy()

# ==========================================
# 📈 3. 產出模型成績單
# ==========================================
mae = mean_absolute_error(true_y, pred_y)

true_direction = true_y > 0
pred_direction = pred_y > 0
accuracy = np.mean(true_direction == pred_direction)

# 載入股票代號
#TENSOR_PATH = '/content/drive/MyDrive/MarketMamba_Database/market_tensors.pt'
#tensor_data = torch.load(TENSOR_PATH, weights_only=False)
#tickers = tensor_data['tickers']
# 👇 換成這個「智慧尋找點名簿」的暴力解法：
num_stocks = pred_y.shape[0]

# 嘗試看看你的 dataset 裡面有沒有自帶點名簿
if hasattr(dataset, 'tickers'):
    tickers = dataset.tickers
else:
    # 如果找不到，我們就直接塞假名字 (Stock_0, Stock_1...) 讓程式能繼續往下跑！
    print("⚠️ 找不到實體點名簿，暫時使用代號 Stock_0 ~ Stock_1936 代替...")
    tickers = [f"Stock_{i}" for i in range(num_stocks)]

df_eval = pd.DataFrame({
    'Ticker': tickers,
    'Predicted_Return': pred_y,
    'True_Return': true_y
})

ic_value = df_eval['Predicted_Return'].corr(df_eval['True_Return'], method='spearman')

top_10_picks = df_eval.sort_values('Predicted_Return', ascending=False).head(10)
actual_profit = top_10_picks['True_Return'].mean()

print("\n==========================================")
print("📝 【MarketMamba 訓練成果總結報告 (測試節點: -100)】 📝")
print("==========================================")
print(f"🥇 資訊係數 (Rank IC): {ic_value:.4f}")
print(f"📏 全市場平均絕對誤差 (MAE): {mae:.4f}")
print(f"🎯 漲跌方向預測準確率:      {accuracy * 100:.2f}%")
print("------------------------------------------")
print("🏆 【當時跟單 Top 10 選股，真實損益結果】")
print(top_10_picks.to_string(index=False))
print("------------------------------------------")
print(f"💰 實戰模擬報酬率: {actual_profit * 100:.2f}% (5天期間)")
print("==========================================")

🔌 啟動 MarketMamba 冷啟動評估中心...

✅ 成功從雲端硬碟載入 MarketMamba 訓練權重！

⏳ 正在預測 8191 號節點 (約 100 個交易日前) 的平行未來...


🌌 回測降噪生成中:   0%|          | 0/100 [00:00<?, ?it/s]

⚠️ 找不到實體點名簿，暫時使用代號 Stock_0 ~ Stock_1936 代替...

📝 【MarketMamba 訓練成果總結報告 (測試節點: -100)】 📝
🥇 資訊係數 (Rank IC): 0.0275
📏 全市場平均絕對誤差 (MAE): 0.0363
🎯 漲跌方向預測準確率:      47.34%
------------------------------------------
🏆 【當時跟單 Top 10 選股，真實損益結果】
    Ticker  Predicted_Return  True_Return
 Stock_510          0.102441    -0.003221
 Stock_795          0.087253    -0.167054
Stock_1618          0.070710    -0.193291
 Stock_390          0.059918    -0.072861
Stock_1760          0.054523     0.002484
  Stock_56          0.054215     0.181147
Stock_1630          0.053643     0.032981
 Stock_774          0.051264    -0.034916
Stock_1674          0.048770    -0.102382
  Stock_78          0.046963    -0.044150
------------------------------------------
💰 實戰模擬報酬率: -4.01% (5天期間)
